<a href="https://colab.research.google.com/github/jaityagi63/ml_scratch/blob/master/hyperparametertunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# parameter vs hyperparameter

## parameter
1. variable model learn from data during model training
2. these variable learn about the underlying pattern of the data

- Ex:
  - coef_ in linear regression
  - weights and bias of the nodes in a neural network

## hyperparameter
1. parameter defined before the model training
2. they help in controlling the learning process and influence the performance of the model

- Ex
  - learning rate and layers in a neural network
  - in lasso the regularisation parameter
- these paramter must be found through trail and error


In [26]:
# training a kneighbour model on the housing data
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score,KFold
from sklearn.neighbors import KNeighborsRegressor

In [27]:
test = pd.read_csv('/content/sample_data/california_housing_test.csv')
train = pd.read_csv('/content/sample_data/california_housing_train.csv')

In [28]:
train.head(4)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0


In [29]:
x = train.iloc[:,:-1]
y = train.iloc[:,-1]

In [30]:
knn = KNeighborsRegressor()
kfold = KFold(n_splits=5, random_state=1,shuffle=True)

score = cross_val_score(knn,x,y,cv=kfold,scoring='r2')

In [31]:
score.mean()

np.float64(0.24951195754962105)

### different ways to tune a model
 - GridSearch Cv ---> perform a exhaustic search over every possible       combination from a param_grid and gives best model using cross validation (   Brute force method)
     - the grid space is called parameter space
 - RandomizedSearchCV --> Instead to trying all possible combiantion what we do is we take random value from parameter space and then train an dwe decide how many time do to that and biggest drawback is that there is a possiblity of never getting best score or best parameter

In [32]:
from sklearn.model_selection import GridSearchCV

In [33]:
""" param_grid is used to make combination think of it like a grid the number of
pramter is like dimension and value of all combination is like cordiante of the grid """

param_grid = {
    'n_neighbors':np.arange(1,10),
    'weights':['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1,2]
}

# grid in the above will of 4d and number of possible conbination is 9*2*4*2 == 144 so the number of model train has to 144


In [34]:
gridcv = GridSearchCV(knn,param_grid,cv=kfold,scoring='r2',refit=True,verbose=2)

In [35]:
gridcv.fit(x,y)

Fitting 5 folds for each of 144 candidates, totalling 720 fits
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=uniform; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=distance; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=distance; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=distance; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=distance; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=1, weights=distance; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p=2, weights=uniform; total time=   0.0s
[CV] END algorithm=auto, n_neighbors=1, p

GridSearchCV(cv=KFold(n_splits=5, random_state=1, shuffle=True),
             estimator=KNeighborsRegressor(),
             param_grid={'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                         'n_neighbors': array([1, 2, 3, 4, 5, 6, 7, 8, 9]),
                         'p': [1, 2], 'weights': ['uniform', 'distance']},
             scoring='r2', verbose=2)

In [36]:
# cv_results_ gives the a dict with key and values from them can get best possible params and comapre them
pd.DataFrame(gridcv.cv_results_)[['param_algorithm','param_n_neighbors','param_p','param_weights','mean_test_score','rank_test_score']].sort_values(by='rank_test_score')

,param_algorithm,param_n_neighbors,param_p,param_weights,mean_test_score,rank_test_score
33,auto,9,1,distance,0.333435,1
69,ball_tree,9,1,distance,0.333435,1
105,kd_tree,9,1,distance,0.333435,1
141,brute,9,1,distance,0.333435,1
68,ball_tree,9,1,uniform,0.329935,5
...,...,...,...,...,...,...
75,kd_tree,1,2,distance,-0.198563,137
2,auto,1,2,uniform,-0.198563,137
3,auto,1,2,distance,-0.198563,137
38,ball_tree,1,2,uniform,-0.198563,137


In [37]:
print(gridcv.best_estimator_)

KNeighborsRegressor(n_neighbors=np.int64(9), p=1, weights='distance')


In [38]:
gridcv.best_score_

np.float64(0.3334349848605937)

In [39]:
gridcv.best_params_

{'algorithm': 'auto',
 'n_neighbors': np.int64(9),
 'p': 1,
 'weights': 'distance'}

In [40]:
from sklearn.model_selection import RandomizedSearchCV

In [41]:
random = RandomizedSearchCV(knn,param_grid,cv=kfold,scoring='r2',refit=True,verbose=3)

In [42]:
random.fit(x,y)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END algorithm=kd_tree, n_neighbors=6, p=2, weights=distance;, score=0.230 total time=   0.0s
[CV 2/5] END algorithm=kd_tree, n_neighbors=6, p=2, weights=distance;, score=0.275 total time=   0.0s
[CV 3/5] END algorithm=kd_tree, n_neighbors=6, p=2, weights=distance;, score=0.291 total time=   0.0s
[CV 4/5] END algorithm=kd_tree, n_neighbors=6, p=2, weights=distance;, score=0.265 total time=   0.0s
[CV 5/5] END algorithm=kd_tree, n_neighbors=6, p=2, weights=distance;, score=0.282 total time=   0.0s
[CV 1/5] END algorithm=brute, n_neighbors=3, p=1, weights=uniform;, score=0.192 total time=   0.7s
[CV 2/5] END algorithm=brute, n_neighbors=3, p=1, weights=uniform;, score=0.235 total time=   0.5s
[CV 3/5] END algorithm=brute, n_neighbors=3, p=1, weights=uniform;, score=0.240 total time=   0.5s
[CV 4/5] END algorithm=brute, n_neighbors=3, p=1, weights=uniform;, score=0.223 total time=   0.5s
[CV 5/5] END algorithm=brute, n_n

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=1, shuffle=True),
                   estimator=KNeighborsRegressor(),
                   param_distributions={'algorithm': ['auto', 'ball_tree',
                                                      'kd_tree', 'brute'],
                                        'n_neighbors': array([1, 2, 3, 4, 5, 6, 7, 8, 9]),
                                        'p': [1, 2],
                                        'weights': ['uniform', 'distance']},
                   scoring='r2', verbose=3)

In [43]:
pd.DataFrame(random.cv_results_)[['param_algorithm','param_n_neighbors','param_p','param_weights','mean_test_score','rank_test_score']].sort_values(by='rank_test_score')

,param_algorithm,param_n_neighbors,param_p,param_weights,mean_test_score,rank_test_score
7,ball_tree,9,1,distance,0.333435,1
8,auto,9,2,distance,0.298663,2
2,ball_tree,5,1,uniform,0.289483,3
3,kd_tree,7,2,uniform,0.278531,4
0,kd_tree,6,2,distance,0.268493,5
5,brute,6,2,distance,0.268493,6
4,auto,6,2,uniform,0.266710,7
1,brute,3,1,uniform,0.226017,8
6,auto,3,2,distance,0.187343,9
9,ball_tree,3,2,distance,0.187343,9


In [44]:
random.best_score_

np.float64(0.3334349848605937)

In [45]:
random.best_params_

{'weights': 'distance',
 'p': 1,
 'n_neighbors': np.int64(9),
 'algorithm': 'ball_tree'}

### for more improved algo use bayesian optimisation
- use libray like:
  - scikit.opt
  - optuna
  - hyperopt
